# 00.1 — Data Ingestion & Baseline Quality Audit

## 1. Mục tiêu
Nạp dữ liệu thô ban đầu (9.617 dòng $\times$ 36 cột) và thực hiện kiểm tra toàn diện chất lượng dữ liệu cơ sở:
1. **Schema & Data Types**: Đảm bảo 35 cột số (`float64`) và 1 cột thời gian (`datetime64`).
2. **Duplicate Records**: Kiểm tra trùng lặp dòng, trùng lặp mốc thời gian.
3. **Invalid Values & Physical Range**: Kiểm tra các giá trị ngoài giới hạn vật lý (ví dụ: `Lượng mưa < 0`, nhiệt độ bất thường).
4. **Baseline Audit**: Chạy `DataQualityAssessment` để ghi nhận hiện trạng ban đầu.
5. **Xuất Artifact**: `outputs/ingestion_audit_report.json`.


In [1]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np

# Reconfigure encoding
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "src").exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.data.loader import DataLoader
from src.data.dataquality import DataQualityAssessment

# 1. Nạp dữ liệu thô (exclude_year_2000=False, drop_redundant_radiation=False)
loader = DataLoader()
df_raw = loader.load_data(exclude_year_2000=False, drop_redundant_radiation=False)

print(f"📊 Dữ liệu thô ban đầu: {df_raw.shape[0]:,} dòng x {df_raw.shape[1]} cột")
assert df_raw.shape == (9617, 36), f"Expected (9617, 36), got {df_raw.shape}"

# 2. Kiểm tra Duplicate
dup_rows = int(df_raw.duplicated().sum())
dup_dates = int(df_raw["Ngày"].duplicated().sum())
print(f"   • Dòng trùng lặp (duplicate rows): {dup_rows}")
print(f"   • Ngày trùng lặp (duplicate dates): {dup_dates}")
assert dup_rows == 0, "Duplicate rows detected!"
assert dup_dates == 0, "Duplicate dates detected!"

# 3. Kiểm tra Invalid Values (Physical Range)
min_rain = float(df_raw["Lượng mưa"].min())
max_rain = float(df_raw["Lượng mưa"].max())
print(f"   • Khoảng giá trị Lượng mưa: min = {min_rain:.2f} mm, max = {max_rain:.2f} mm")
assert min_rain >= 0.0, "Invalid negative rainfall detected!"

# 4. Kiểm tra kiểu dữ liệu
datetime_valid = pd.api.types.is_datetime64_any_dtype(df_raw["Ngày"])
numeric_cols_count = len(df_raw.select_dtypes(include=[np.number]).columns)
print(f"   • Cột Ngày hợp lệ (datetime64): {datetime_valid}")
print(f"   • Số cột dạng số: {numeric_cols_count} / 35")
assert datetime_valid, "Datetime column invalid!"
assert numeric_cols_count == 35, "Numeric column count mismatch!"


📊 Dữ liệu thô ban đầu: 9,617 dòng x 36 cột
   • Dòng trùng lặp (duplicate rows): 0
   • Ngày trùng lặp (duplicate dates): 0
   • Khoảng giá trị Lượng mưa: min = 0.00 mm, max = 143.31 mm
   • Cột Ngày hợp lệ (datetime64): True
   • Số cột dạng số: 35 / 35


In [2]:
# 5. Chạy Baseline Audit qua DataQualityAssessment
assessor = DataQualityAssessment(df_raw, target_col="Lượng mưa", mode="notebook")
quality_results = assessor.assess_quality(datetime_col="Ngày")

# 6. Xuất Ingestion Audit Artifact
OUTPUT_DIR = ROOT_DIR / "notebooks" / "data_quality" / "outputs"
ingestion_artifact = {
    "total_raw_rows": df_raw.shape[0],
    "total_raw_columns": df_raw.shape[1],
    "date_range": {
        "start": str(df_raw["Ngày"].min().date()),
        "end": str(df_raw["Ngày"].max().date())
    },
    "duplicate_rows": dup_rows,
    "duplicate_dates": dup_dates,
    "target_variable": "Lượng mưa",
    "target_min": min_rain,
    "target_max": max_rain,
    "total_missing_cells_baseline": int(df_raw.isna().sum().sum()),
    "columns_with_missing_count": int((df_raw.isna().sum() > 0).sum())
}

artifact_path = OUTPUT_DIR / "ingestion_audit_report.json"
with open(artifact_path, "w", encoding="utf-8") as f:
    json.dump(ingestion_artifact, f, indent=2, ensure_ascii=False)

print(f"\n💾 Ingestion audit report exported to: {artifact_path}")
print(f"   • Tổng ô thiếu ban đầu: {ingestion_artifact['total_missing_cells_baseline']} ô")
print(f"   • Số cột có missing ban đầu: {ingestion_artifact['columns_with_missing_count']} cột")



DATA QUALITY ASSESSMENT
Dataset shape: 9,617 rows x 36 cols
Missing data: 1,830 / 346,212 cells (0.53%) across 5 column(s)
    - Bức xạ trực tiếp pháp tuyến: 366 missing (3.81%)
    - Bức xạ quang hợp trời quang: 366 missing (3.81%)
    - Bức xạ UVA: 366 missing (3.81%)
    - Bức xạ UVB: 366 missing (3.81%)
    - Chỉ số UV: 366 missing (3.81%)
Duplicate rows: 0 (0.0%)
Data types: {'float64': 35, 'datetime64[ns]': 1}
Target 'Lượng mưa': missing=0 (0.0%), zero=2,468 (25.66%), negative(invalid)=0 (0.0%), range=[0.00, 143.31], mean=3.95
Datetime 'Ngày': unparsable=0, duplicate_dates=0, gaps_detected=0

💾 Ingestion audit report exported to: D:\DS108_project\notebooks\data_quality\outputs\ingestion_audit_report.json
   • Tổng ô thiếu ban đầu: 1830 ô
   • Số cột có missing ban đầu: 5 cột
